# T033 · Molecular properties: multitask modelling

**Note:** This talktorial is not part of the official TeachOpenCADD, but uses its template and has been developed as part of the ACMDD (Advanced Computational Methods in Drug Discovery), MSc course, Leiden University, 2026.

Authors:

- Ida de Vries, 2026, [Computational Drug Discovery](https://www.universiteitleiden.nl/en/science/drug-research/drug-discovery-and-safety/computational-drug-discovery), Drug Discovery & Safety Leiden University (The Netherlands)
- Olivier J.M. Béquignon, 2026, Computational Drug Discovery, Drug Discovery & Safety Leiden University (The Netherlands)


This talktorial is the direct continuation of **Talktorial T032**: it reuses its modelling scaffolding and replaces its endpoint.

## Aim of this talktorial

In **Talktorial T032** we built a proteochemometrics (PCM) model: a *single* model covering *several* protein targets, in which the target identity entered the model as an extra block of input features (the protein descriptors). That is one way of making one model serve many related prediction problems.

This talktorial explores the other way: **multitask modelling**, where the several problems become several *outputs* of one model instead of extra *inputs*. And instead of predicting how strongly a molecule binds a protein, we predict what the molecule *is like* - its physicochemical properties: melting point, lipophilicity, aqueous solubility, and so on. These are the properties that decide whether a promising hit can ever be formulated, dosed and absorbed, and they are exactly the properties that a medicinal chemist must trade off against potency.

Our data come from the [OCHEM](https://ochem.eu) database (the *Online CHEmical Modeling environment*), one of the largest curated collections of experimental physicochemical and toxicological measurements.

Along the way we will meet the three ways of getting several numbers out of one scikit-learn-style model - estimators with **native** multi-output support, the **`MultiOutputRegressor`/`MultiOutputClassifier` wrappers**, and **XGBoost's `multi_strategy='multi_output_tree'`** - and we will take data splitting much more seriously than a single random split, because with several endpoints there are several ways to fool yourself at once.

### Contents in *Theory*

* From proteochemometrics to multitask modelling
* Data preparation
    * The OCHEM database
    * The properties we model, and why they belong together
    * Molecule encoding: molecular descriptors
* Multitask machine learning
    * Why sharing a model between tasks can help - and when it hurts
    * Three implementations of multi-output learning
        * Estimators with native multi-output support
        * The `MultiOutputRegressor` / `MultiOutputClassifier` wrappers
        * Gradient boosting: `multi_strategy='one_output_per_tree'` vs `'multi_output_tree'`
    * Missing labels: the sparse label matrix
* Data splitting methods
    * Random, scaffold, cluster and stratified splits
    * Model selection vs. model assessment: cross-validation on top of a split
    * Repeating a split: how much of a difference is a real difference?
* Evaluation metrics for multi-output models

### Contents in *Practical*

* Load the OCHEM property data
* Data preparation
    * Exploring the endpoints
    * From long to wide: the label matrix
    * How much do the tasks overlap, and how correlated are they?
    * Calculate compound descriptors
    * Building the feature matrix (and avoiding a descriptor that gives the game away)
    * How many tasks can we afford?
* Data splitting
    * Grouping compounds: scaffolds and similarity clusters
    * The splitter
    * Looking at what each split actually did
    * Quantifying the leakage: nearest-neighbour similarity between test and training set
* Multitask regression
    * Helper functions
    * The models: native, wrapped and boosted
    * A first model, on a single split
    * The full comparison: models × splits × repeats
    * Model selection done properly: cross-validation that matches the split
    * Does multitask learning actually help? A learning-curve experiment
    * Using *all* the data: what the wrappers can do that joint models cannot
* Multitask classification
    * Turning the endpoints into developability flags
    * Three multi-label classifiers: `MultiOutputClassifier`, native multi-label forests, XGBoost
    * Metrics for multi-label problems
* Discussion
* Quiz

### References

* OCHEM database: [*J. Comput. Aided Mol. Des.*, **25**, 533–554 (2011)](https://doi.org/10.1007/s10822-011-9440-2) and [ochem.eu](https://ochem.eu)
* Multitask learning, the original argument: Caruana, [*Machine Learning*, **28**, 41–75 (1997)](https://doi.org/10.1023/A:1007379606734)
* Multitask networks for QSAR: [*J. Chem. Inf. Model.*, **55**, 263–274 (2015)](https://doi.org/10.1021/ci500747n)
* Multi-output decision trees: [*Machine Learning*, **73**, 185–214 (2008)](https://doi.org/10.1007/s10994-008-5077-3)
* scikit-learn, multiclass and multioutput algorithms [(Docs)](https://scikit-learn.org/stable/modules/multiclass.html)
* XGBoost, multiple outputs [(Docs)](https://xgboost.readthedocs.io/en/stable/tutorials/multioutput.html)
* Scaffold splits: Bemis & Murcko, [*J. Med. Chem.*, **39**, 2887–2893 (1996)](https://doi.org/10.1021/jm9602928)
* On splitting and over-optimism in QSAR: [*J. Chem. Inf. Model.*, **58**, 916–932 (2018)](https://doi.org/10.1021/acs.jcim.7b00403)
* The General Solubility Equation: [*J. Pharm. Sci.*, **69**, 912–922 (1980)](https://doi.org/10.1002/jps.2600690814)
* Regression and classification metrics [(scikit-learn Docs)](https://scikit-learn.org/stable/modules/model_evaluation.html)

## Theory

### From proteochemometrics to multitask modelling

In **Talktorial T032** we wanted one model to cover four adenosine receptors. The trick there was to *describe* the protein: every row of the training table was a compound–protein pair, the features were `[protein descriptors | molecular descriptors]`, and there was a single output column, the pChEMBL value. The model learned a function

$$ \hat{y} = f(\mathbf{x}_{\text{compound}},\ \mathbf{x}_{\text{protein}}) $$

That approach has a great strength: because the protein enters as a *feature vector*, the model can in principle extrapolate to a protein it has never seen, as long as that protein's descriptor lands in a region it understands. That is exactly what the leave-one-target-out (LOTO) split tested.

Now suppose the several prediction problems are not four receptors but four *physicochemical properties*. There is no useful "property descriptor" - melting point and logP are not points in a shared space the way two homologous receptors are. So the PCM trick does not apply. What we can do instead is give the model several output columns:

$$ (\hat{y}_1, \hat{y}_2, \ldots, \hat{y}_T) = f(\mathbf{x}_{\text{compound}}) $$

This is **multitask learning**: one model, one shared internal representation of the molecule, $T$ heads reading off $T$ answers (i.e., properties in this example).

| | Proteochemometrics (T032) | Multitask (this talktorial) |
|---|---|---|
| One row is | a compound–target pair | a compound |
| The "task" is | encoded in the input features | a column of the output |
| Number of outputs | 1 | $T$ |
| Can handle a brand-new task | yes, if it can be described | no - a new property needs a retrained model |
| Needs the tasks to live in one descriptor space | yes | no |
| Sharing happens because | the same weights see all pairs | the tasks share the representation |

<img src='images/multitask_concept.png' width="750">

*Figure 1:*
Single-task modelling (left) fits one independent model per property, each on the compounds measured for that property. Multitask modelling (right) fits one model with a shared internal representation and one output per property.

### Data preparation

#### The OCHEM database

[OCHEM](https://ochem.eu), the *Online CHEmical Modeling environment*, is a web platform that combines two things: a database of experimental measurements collected from the literature, and a modelling framework built on top of it ([*J. Comput. Aided Mol. Des.*, **25**, 533–554 (2011)](https://doi.org/10.1007/s10822-011-9440-2)). At the time of writing it holds several million records covering several hundred properties, gathered from more than twenty thousand sources, and contributed data are released under a Creative Commons CC-BY licence.

What makes OCHEM interesting for us, compared with a bioactivity resource such as ChEMBL (**Talktorial T001**) or Papyrus (**Talktorial T032**), is its coverage of *physicochemical and ADME/Tox endpoints*: melting and boiling points, partition and distribution coefficients, water solubility, vapour pressure, pKa, plasma protein binding, Ames mutagenicity, and many more. Each record is tied to its literature source, its unit and - where relevant - its measurement conditions.

> **Note**
>
> Unlike the ChEMBL, PubChem and Papyrus resources used in earlier talktorials, **OCHEM requires a (free) user account** and does not offer an anonymous programmatic interface. A notebook therefore cannot download its data on the fly. The exports used here were downloaded once from the OCHEM properties browser, cleaned with `scripts/prepare_ochem_data.py`, and the small result was committed to `data/`. This is the same pattern Talktorial T032 uses for its pre-computed ClustalO alignment. See `data/README.md` for exactly which property sets were exported and how they were processed.

#### The properties we model, and why they belong together

A multitask model is only worth building if the tasks have something to say to each other. Otherwise, each head would confuse the other and the shared representation they rely on would be sub-optimal. Picking endpoints at random and stacking them into one output matrix is a good way to get a model that is worse than $T$ separate ones.

The endpoints used here were chosen because they are genuinely, *physically* coupled. The clearest statement of that coupling is the **General Solubility Equation** (GSE, [*J. Pharm. Sci.*, **69**, 912–922 (1980)](https://doi.org/10.1002/jps.2600690814)), an empirical relationship that predicts aqueous solubility from just two other measurable quantities:

$$ \log S \approx 0.5 - 0.01\,(\mathrm{MP} - 25) - \log P $$

where $\log S$ is the log molar solubility, $\mathrm{MP}$ the melting point in °C, and $\log P$ the octanol/water partition coefficient. Read as chemistry rather than as a formula, it says: *a compound is poorly soluble in water either because it would rather sit in oil (high $\log P$), or because it is locked into a stable crystal (high melting point)*. Those are two different physical causes, and the GSE separates them.

So the three endpoints are not three independent questions: knowing two of them very nearly determines the third. A model asked to predict all three at once is being pushed towards an internal representation that respects that relationship - which is precisely the kind of inductive bias multitask learning is supposed to exploit. Whether it *actually* does is the question the practical half of this talktorial measures rather than assumes — and the answer is not the one the motivation leads you to expect. Boiling point and vapour pressure add a second, related family: both are governed by how strongly the molecules hold on to each other in the condensed phase, the same intermolecular forces that set the melting point.

The endpoints also differ a lot in *difficulty*, which will turn out to be useful:

* **logP** is the easy one. It is close to additive in atom and fragment contributions, which is why simple methods such as Crippen's atom-contribution model already work well. Treat it as the sanity check: if a model cannot do logP, something is wrong.
* **Melting point** is notoriously hard. It depends on crystal packing - on how molecules arrange themselves in the solid state - and that is a property of the *crystal*, not of the isolated molecule that our descriptors describe. Published models rarely get far past $R^2 \approx 0.6$–$0.7$, and errors of 30–40 °C are normal.
* **Water solubility** sits in between, and inherits some of the melting point's difficulty through the GSE.

#### Molecule encoding: molecular descriptors

As in **Talktorial T032**, we encode each molecule as a vector of **molecular descriptors** computed with RDKit: molecular weight, topological polar surface area, ring counts, connectivity indices, counts of hydrogen-bond donors and acceptors, and so on. Descriptors (rather than the fingerprints of **Talktorial T007**) are the natural choice for physicochemical endpoints, because many of them are direct numerical summaries of the very effects that drive those endpoints - polar surface area and hydrogen bonding drive solubility; size and flexibility drive boiling point.

There is one trap worth naming in advance. RDKit's descriptor list includes `MolLogP` and `MolMR`, which are *not* measurements or structural counts: they are the output of Crippen's atom-contribution **model** of logP and molar refractivity. Leaving `MolLogP` in the feature matrix while asking the model to predict experimental logP would not be leakage in the strict sense - no experimental value crosses from test to train - but it would reduce the logP task to "learn to correct Crippen", and it would flatter our results for reasons that have nothing to do with multitask learning. We therefore remove these two from the features, and instead keep Crippen's `MolLogP` aside as an honest **baseline to beat**.

### Multitask machine learning

#### Why sharing a model between tasks can help - and when it hurts

Training one model on several related tasks can beat training one model per task for a few distinct reasons ([Caruana, *Machine Learning*, **28**, 41–75 (1997)](https://doi.org/10.1023/A:1007379606734)):

* **More data for the shared part.** Every task contributes to learning which regions of descriptor space matter. A task with 800 measurements gets to piggyback on the representation learned from a task with 40,000.
* **Regularisation.** A representation that must serve several endpoints cannot overfit the quirks of any one of them. Multitask learning acts as a data-dependent prior.
* **Eavesdropping.** Some feature may be hard to discover from task A alone but easy from task B; once found, task A can use it. The GSE says exactly this about melting point and solubility.
* **Implicit data augmentation.** The noise in different endpoints is largely independent, so averaging the pressure from several tasks can cancel part of it.

It is not free, though. Multitask learning goes wrong when:

* **The tasks are unrelated.** The model then spends capacity on a compromise representation that serves nobody - *negative transfer*. A joint melting-point-and-hERG-blockade model is asking for it.
* **The tasks are on wildly different scales.** A squared-error objective summed over outputs is dominated by whichever task has the largest variance. Melting point in °C has a variance thousands of times that of logP in log units. **Standardising each output before fitting is not optional.**
* **The label counts are very unbalanced.** The abundant task drives the shared structure and the rare one is along for the ride.
* **The labels are sparse** - the subject of its own section below.

#### Three implementations of multi-output learning

<img src='images/multitask_strategies.png' width="900">

*Figure 2:*
Three ways to obtain several outputs. **A**: the `MultiOutput*` wrappers clone one estimator per task and fit each clone on one column of `Y` - nothing is shared. **B**: estimators with native multi-output support build one structure whose splits are scored on the impurity summed over all tasks, and whose leaves store a vector. **C**: XGBoost can either grow one tree per task per boosting round (`'one_output_per_tree'`, the default) or a single tree per round with vector-valued leaves (`'multi_output_tree'`).

##### Estimators with native multi-output support

Several scikit-learn estimators accept a 2-D `y` of shape `(n_samples, n_outputs)` directly, and mean it: they build **one** model for all outputs.

For the tree-based ones (`DecisionTreeRegressor`, `RandomForestRegressor`, `ExtraTreesRegressor`) the change is small and elegant. A single-output regression tree scores a candidate split by how much it reduces the variance of $y$. A multi-output tree scores it by how much it reduces the variance **summed over all output columns**, and each leaf stores the vector of per-task means of the samples that reached it. The tree structure - every split, in every node - is therefore shared by all tasks, and each split is a compromise that has to earn its place across the board. That shared structure is where the transfer between tasks comes from.

Other natively multi-output estimators arrive at it differently:

* `KNeighborsRegressor` - the neighbours of a query point do not depend on which output you ask about, so returning a vector is free.
* `RidgeCV`, `LinearRegression`, `PLSRegression` - a linear model with a matrix of coefficients; with a plain squared-error loss this is mathematically identical to $T$ separate fits, but `PLSRegression` genuinely shares, because the latent components it extracts are chosen to explain all responses at once.
* `MLPRegressor` - a neural network with $T$ output neurons: all hidden layers are shared, which is the textbook multitask architecture (compare the networks in **Talktorials T021 and T022**).

##### The `MultiOutputRegressor` / `MultiOutputClassifier` wrappers

Most estimators do *not* accept a 2-D `y`: `SVR`, `GradientBoostingRegressor`, `HistGradientBoostingRegressor`, `LogisticRegression`, `SVC` and many more are strictly single-output. `sklearn.multioutput.MultiOutputRegressor` and `MultiOutputClassifier` make them usable anyway, by cloning the estimator $T$ times and fitting clone $t$ on column $t$ of `Y`.

Be clear about what this does and does not buy you. It buys **convenience**: one object, one `fit`, one `predict` returning an `(n_samples, T)` array, one estimator that drops into a `Pipeline` or a `GridSearchCV`. It does **not** buy transfer. The clones never see each other's labels; the predictions are numerically identical to fitting $T$ models in a `for` loop. That makes the wrapper the perfect *control experiment*: run the same base estimator wrapped and natively multi-output, and any difference you see is the effect of sharing, not of the algorithm.

(There is a genuinely different wrapper, `RegressorChain` / `ClassifierChain`, which feeds the prediction of task 1 as an extra feature to task 2, and so on down a chain. That *does* transfer information, at the price of caring about the order of the chain and of accumulating errors along it.)

##### Gradient boosting: `one_output_per_tree` vs `multi_output_tree`

XGBoost supports multiple outputs natively, and exposes the choice we have been drawing out as a single parameter, `multi_strategy`:

* **`'one_output_per_tree'`** (the default) - at every boosting round, grow $T$ separate trees, one per output. This is the wrapper strategy, folded into the booster: accurate, but the model has $T \times$ as many trees, and nothing is shared.
* **`'multi_output_tree'`** - at every boosting round, grow **one** tree whose leaves hold a vector of $T$ values. When a candidate split is scored, the gradient and Hessian statistics of all outputs are summed, so the split has to be good for the ensemble of tasks.

The second option gives smaller, faster models that share structure across outputs, and it is the boosting analogue of the native multi-output forest. The XGBoost documentation notes that it is a newer feature and that its accuracy on any given problem is an empirical question - which is precisely the sort of question a talktorial should answer by measuring it.

Note also `tree_method='hist'` is required for `'multi_output_tree'`, and that the same parameter works for `XGBRegressor` and for multi-label `XGBClassifier`.

#### Missing labels: the sparse label matrix

Here is the practical problem that dominates real multitask work on public data. In a bioactivity matrix, a missing entry means "this compound was never tested against this target". In a physicochemical matrix it means "nobody ever measured this property for this compound" - and since melting points, logP values and solubilities were measured by different communities for different reasons, most of the matrix is empty.

<img src='images/sparse_label_matrix.png' width="900">

*Figure 3:*
The label matrix of a multitask property dataset is sparse. Keeping only complete cases (middle) is simple but throws away most of the data. Masking (right) keeps everything but needs a model that can ignore the gaps.

This matters because **neither scikit-learn's multi-output estimators nor XGBoost's multi-output trees accept `NaN` in `y`**. Both raise an error. (Missing values in `X` are a different story: `HistGradientBoostingRegressor` and XGBoost handle those natively.) So a multi-output model forces a decision:

1. **Complete cases.** Keep only the compounds measured for every task. Simple, correct, and usually very expensive - the intersection of $T$ sparse sets shrinks fast.
2. **Impute the gaps.** Fill missing labels with, say, the training mean. Cheap, but you are now training the model to predict a number nobody measured, and it will drag every prediction towards the mean.
3. **Mask per task.** Fit each task only on the rows that have a label for it. This keeps all the data - but it is only possible if the tasks are fitted separately, i.e. with the wrapper strategy, or with a model that supports per-sample-per-output weights (a neural network with a masked loss is the standard answer, and it is what deep multitask QSAR models such as those in [*J. Chem. Inf. Model.*, **55**, 263–274 (2015)](https://doi.org/10.1021/ci500747n) do).

There is a real tension here, and it is the central trade-off of this talktorial: **the models that share the most are exactly the ones that demand a dense label matrix.** We will measure both sides of it.

### Data splitting methods

**Talktorial T032** introduced random, leave-one-target-out, leave-one-compound-cluster-out, temporal and stratified splits. The logic carries over, with one change: there are no protein targets to leave out any more, so the interesting axis is entirely chemical. A physicochemical dataset assembled from decades of literature is also *full* of near-duplicates - homologous series, one dataset re-published in another - which makes a random split more misleading here than almost anywhere else.

<img src='images/splitting_methods.png' width="900">

*Figure 4:*
Four ways of splitting the same set of compounds. Each panel shows the same toy chemical space; blue circles are training compounds and orange diamonds test compounds.

* **Random split.** Every compound is assigned independently. This is the baseline, and it is *optimistic*: if a training compound and a test compound differ by one methyl group, the model is being asked an easy question that it will almost always get right. Useful as a point of comparison and for quick iteration; never as a claim about how a model will behave on new chemistry.
* **Scaffold split.** Compounds are grouped by their **Bemis–Murcko scaffold** - the ring systems and the linkers between them, with the side chains stripped off ([*J. Med. Chem.*, **39**, 2887–2893 (1996)](https://doi.org/10.1021/jm9602928)) - and whole scaffolds go to one side or the other. This answers: *can the model handle a core structure it has never seen?*
* **Cluster split.** Compounds are clustered by fingerprint similarity (as in **Talktorial T005**) and whole clusters are held out. Related in spirit to the scaffold split, but it groups on overall similarity rather than on the ring system, so it also separates compounds that share no scaffold but are nonetheless near-identical.
* **Stratified random split.** Still random, but constrained so that the distribution of the endpoint is the same on both sides. For classification this means preserving the class balance - essential when a class is rare. For regression, one bins the values and stratifies on the bins. With $T$ tasks you can only stratify exactly on one thing at a time, which is itself worth noticing.

**Why we use several splits rather than picking one.** Each split answers a different question, and a model that wins under one can lose under another. Reporting a single number from a single random split is the single most common way that published QSAR/QSPR performance turns out to be unreproducible in practice ([*J. Chem. Inf. Model.*, **58**, 916–932 (2018)](https://doi.org/10.1021/acs.jcim.7b00403)). The spread across splitting strategies *is* the result: the random-split score tells you how well the model interpolates within the chemistry it knows, and the cluster-split score tells you what to expect on a genuinely new series. The gap between them is a measurement of how much of your apparent performance was near-duplicate matching.

#### Model selection vs. model assessment

Two different jobs get called "validation", and conflating them is how test sets quietly stop being test sets.

* **Model assessment** - estimating how well a *finished* model will do on new data. This is what the held-out test set is for, and it may be looked at once, at the end.
* **Model selection** - choosing between models, or tuning hyperparameters. This must happen using the training set only, typically by **cross-validation**: split the training set into $k$ folds, fit on $k-1$ and score on the held-out one, then rotate, and finally average.

The subtlety for us is that cross-validation inherits the split question. If the test set was built by holding out whole scaffolds but hyperparameters were tuned by *random* k-fold cross-validation inside the training set, then the tuning was optimised for a question different from the one being asked at the end. The fix is `GroupKFold` (or scikit-learn's `StratifiedGroupKFold`): pass the scaffold or cluster label as `groups` and the folds respect it. **The cross-validation scheme should mirror the test split.**

#### Repeating a split: how much of a difference is a real difference?

One more habit worth building. A single random split produces a single number, and that number has a standard error that nobody printed. On a test set of a few hundred compounds, two models whose $R^2$ differ by 0.02 are, in all likelihood, indistinguishable.

The cheap fix is to **repeat the split with different random seeds** and report the mean and spread. Everything below is run over several repeats for exactly this reason, and the error bars are the point of the plots, not decoration. When a difference is smaller than the spread across repeats, the honest summary is "no measurable difference" - which is a perfectly good scientific result, and a much better one than a ranking built out of noise.

### Evaluation metrics for multi-output models

The regression metrics are the ones from **Talktorial T032** - coefficient of determination $R^2$, Pearson's $r$, mean absolute error (MAE), root-mean-square error (RMSE) - but the multi-output setting adds a question: *how do you get from $T$ numbers to one?*

scikit-learn's `r2_score` takes a `multioutput` argument that spells out the choices:

* `'raw_values'` - do not aggregate; return one score per task. **This should be your default.** It is the only option that lets you see negative transfer, which typically shows up as one task getting worse while the others improve.
* `'uniform_average'` - the plain mean over tasks. Fine for a headline number, as long as the per-task values are reported next to it.
* `'variance_weighted'` - weight each task by the variance of its true values. This sounds principled and is usually a trap: it lets the highest-variance endpoint (here, melting point in °C) dominate the summary.

Two further points specific to multitask work:

* **Never average metrics computed on different scales.** An MAE of 33 (melting point, °C) and an MAE of 0.55 (logP, log units) do not have a mean. $R^2$ is dimensionless and *can* be averaged; absolute error metrics must be reported per task, in their own units.
* **Compare against the right baseline.** $R^2 = 0$ corresponds to always predicting the training mean. For a task with a narrow spread this is already a decent predictor, so a modest $R^2$ there may be worth more than a large $R^2$ on a wide-spread task. Where a domain baseline exists - Crippen's `MolLogP` for logP, or the GSE for solubility - beating *that* is the meaningful test.

For the classification half we will use **Matthews correlation coefficient (MCC)** and **balanced accuracy**, both of which stay honest when classes are imbalanced, plus **ROC-AUC** on the predicted probabilities. Multi-label problems add two aggregate views: **Hamming loss** (the fraction of individual labels predicted wrongly) and **subset accuracy** (the fraction of compounds for which *every* label was right) - the latter is a deliberately harsh measure, and it drops quickly as $T$ grows.

## Practical

In the practical part we build multitask models for a set of physicochemical endpoints taken from OCHEM, compare the three multi-output implementations against single-task baselines, and do all of it under four different data splits so that we can see which of our conclusions survive.

`xgboost` is not part of the default environment, so we install it here. Everything else (RDKit, scikit-learn, pandas, seaborn) is already available.

In [ ]:
!pip install -q xgboost

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.ML.Cluster import Butina

from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import RidgeCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    hamming_loss,
    matthews_corrcoef,
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, KFold, cross_val_score, train_test_split
from sklearn.multioutput import MultiOutputClassifier, MultiOutputRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR

from xgboost import XGBClassifier, XGBRegressor

from tqdm.auto import tqdm

RDLogger.DisableLog("rdApp.*")
sns.set_theme(style="whitegrid", context="notebook")

As in **Talktorial T007**, we fix the random seed so that the notebook gives the same numbers every time it is run. Note that this is about *reproducibility*, not about correctness: a result that only holds for one seed is not a result, which is why several of the experiments below are repeated over a handful of seeds.

In [ ]:
SEED = 22
rng = np.random.default_rng(SEED)

# Set path to this notebook
HERE = Path(_dh[-1])
DATA = HERE / "data"

### Load the OCHEM property data

The file `data/ochem_properties.csv.gz` is a **long/tidy** table: one row per (compound, property) pair. It was produced by `scripts/prepare_ochem_data.py` from raw OCHEM exports; that script reads OCHEM's unit-converted value column, standardises the structures with RDKit (largest organic fragment, neutralised, stereochemistry removed), discards unusable records (unparsable SMILES, inorganics, values outside a physically sensible range) and aggregates the remaining literature records per compound into a median. See `data/README.md` for the details, including the two places where a unit had to be derived.

Keeping it in long format is deliberate. It is the format the database gives you, it holds the sparsity honestly (a missing measurement is simply an absent row rather than a `NaN` you have to remember to handle), and turning it into the wide matrix a multitask model needs is a step worth doing explicitly.

Two columns deserve a note:

* `n_records` — how many independent literature records were aggregated into this value.
* `value_std` — the standard deviation across those records, i.e. **how much the literature disagrees with itself**. This is our estimate of the experimental noise floor, and it is the number a model's error should really be compared against. It is empty for compounds measured only once.

In [ ]:
ochem = pd.read_csv(DATA / "ochem_properties.csv.gz")
print(f"{len(ochem):,} compound-property pairs")
print(f"{ochem['inchikey'].nunique():,} unique compounds")
ochem.head()

In [ ]:
# What is in the file, in which unit, and how much of it?
replicated = ochem[ochem["n_records"] > 1]

summary = (
    ochem.groupby("property")
    .agg(
        unit=("unit", "first"),
        compounds=("inchikey", "nunique"),
        median_value=("value", "median"),
        spread_p5_p95=("value", lambda s: f"{s.quantile(0.05):.1f} to {s.quantile(0.95):.1f}"),
    )
)
# how much the literature disagrees with itself, over the compounds that were
# measured more than once
summary["replicated compounds"] = replicated.groupby("property").size()
summary["lit. sd (median)"] = replicated.groupby("property")["value_std"].median()
summary["lit. sd (p90)"] = replicated.groupby("property")["value_std"].quantile(0.90)
summary = summary.sort_values("compounds", ascending=False).round(3)
summary

The last three columns are worth pausing on. `lit. sd` is the standard deviation across independent literature records of the *same* property for the *same* compound — our estimate of the experimental noise floor. No model built on this data can be expected to beat it.

The **median** is close to zero for most endpoints, and that is a finding rather than a reassurance: it means that for over half the replicated compounds every source reports *exactly* the same number. Literature compilations copy from one another, so "five independent records" often means one measurement propagated five times. The **90th percentile** is the more honest figure — a few degrees for melting point, a couple of tenths of a log unit for logP and logS — and that is the bar to compare a model's MAE against.

### Data preparation

#### Exploring the endpoints

In [ ]:
# Order the tasks by how much data they have, and keep the ordering fixed
# everywhere in the notebook so that colours stay attached to endpoints.
# The units are those of the OCHEM export (see data/README.md).
TASK_LABELS = {
    "melting_point": "Melting point (°C)",
    "logp": "logP (log unit)",
    "logs": "logS (log mol/L)",
    "boiling_point": "Boiling point (°C)",
    "vapour_pressure": "log vapour pressure (log Pa)",
}
TASKS = [t for t in TASK_LABELS if t in set(ochem["property"])]
print("tasks:", TASKS)

In [ ]:
g = sns.displot(
    ochem,
    x="value",
    row="property",
    row_order=TASKS,
    element="step",
    kde=True,
    height=1.9,
    aspect=3.2,
    facet_kws={"sharex": False, "sharey": False},
)
for ax, task in zip(g.axes.flat, TASKS):
    ax.set_title(TASK_LABELS[task], fontsize=11)
    ax.set_xlabel("")
g.figure.suptitle("Distribution of each endpoint", y=1.02)
plt.show()

The distributions are worth reading before modelling anything. Melting and boiling points are broad and roughly symmetric, in degrees Celsius; logP, logS and vapour pressure are narrow and in log units. The scales differ by two to three orders of magnitude - the concrete reason why, as promised in the theory section, we will standardise every output before fitting a joint model. Without that step, a squared-error objective summed over the outputs would be almost entirely a melting-point objective.

#### From long to wide: the label matrix

In [ ]:
# One row per compound, one column per task. Missing measurements become NaN.
labels = ochem.pivot(index="inchikey", columns="property", values="value")[TASKS]

# Keep the structure of each compound alongside it
structures = (
    ochem.drop_duplicates("inchikey").set_index("inchikey")["smiles"].reindex(labels.index)
)

print(f"label matrix: {labels.shape[0]:,} compounds x {labels.shape[1]} tasks")
print(f"filled: {labels.notna().to_numpy().mean():.1%} of the matrix")
labels.head()

In [ ]:
# How many compounds have how many measurements?
n_measured = labels.notna().sum(axis=1)
counts = n_measured.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6.5, 3.2))
bars = ax.bar(counts.index, counts.values, color="#2a78d6", width=0.65)
ax.bar_label(bars, fmt="%d", padding=3, fontsize=9)
ax.set_xlabel("number of endpoints measured for a compound")
ax.set_ylabel("compounds")
ax.set_xticks(counts.index)
ax.set_title("Most compounds are measured for only a few endpoints")
sns.despine()
plt.show()

complete_cases = labels.dropna()
print(f"compounds with all {len(TASKS)} endpoints: {len(complete_cases):,} "
      f"({len(complete_cases) / len(labels):.1%} of the set)")

This is the sparsity of Figure 3, in numbers. Insisting on complete cases costs us most of the dataset - and it would cost more still if we added a sixth endpoint. That is the price of a joint model, and we will measure what it buys in return.

#### How much do the tasks overlap, and how correlated are they?

In [ ]:
# Pairwise co-measurement: how many compounds have both endpoint i and endpoint j?
present = labels.notna().astype(int)
overlap = pd.DataFrame(
    present.T.to_numpy() @ present.to_numpy(), index=TASKS, columns=TASKS
)

# Pairwise correlation, computed on whatever compounds the pair has in common
correlation = labels.corr(method="pearson", min_periods=30)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
short = [TASK_LABELS[t].split(" (")[0] for t in TASKS]

sns.heatmap(
    overlap, annot=True, fmt=",d", cmap="Blues", cbar=False,
    xticklabels=short, yticklabels=short, ax=axes[0],
    annot_kws={"fontsize": 9},
)
axes[0].set_title("Compounds measured for both endpoints")

sns.heatmap(
    correlation, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0,
    xticklabels=short, yticklabels=short, ax=axes[1],
    annot_kws={"fontsize": 9},
)
axes[1].set_title("Pearson correlation between endpoints\n(pairwise-complete compounds)")

for ax in axes:
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.show()

The left panel is the practical constraint - which pairs of endpoints there is enough shared data to learn anything joint about. The right panel is the reason to try: endpoints that correlate strongly have something to tell each other, and are the ones from which we should expect transfer. Endpoints near zero correlation are the negative-transfer risk.

Note the sign conventions: solubility is expected to correlate *negatively* with both logP and melting point (that is the General Solubility Equation), and vapour pressure negatively with boiling point (what boils at a high temperature does not evaporate at room temperature).

Since the General Solubility Equation was our justification for putting these endpoints in one model, we should check that it actually holds in *this* data before relying on it.

In [ ]:
gse_subset = labels[["melting_point", "logp", "logs"]].dropna()

if len(gse_subset) >= 50:
    gse_prediction = (
        0.5 - 0.01 * (gse_subset["melting_point"] - 25) - gse_subset["logp"]
    )
    observed = gse_subset["logs"]

    fig, ax = plt.subplots(figsize=(4.8, 4.6))
    ax.scatter(gse_prediction, observed, s=12, alpha=0.35, color="#2a78d6",
               edgecolor="none")
    lims = (min(gse_prediction.min(), observed.min()) - 0.5,
            max(gse_prediction.max(), observed.max()) + 0.5)
    ax.plot(lims, lims, color="#52514e", lw=1.4, ls="--", label="perfect agreement")
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel("logS predicted by the GSE\n(from melting point and logP)")
    ax.set_ylabel("logS measured")
    ax.set_title(f"The General Solubility Equation on our data\n(n = {len(gse_subset):,})")
    ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f"GSE:  R2 = {r2_score(observed, gse_prediction):.3f}   "
          f"MAE = {mean_absolute_error(observed, gse_prediction):.3f} log units   "
          f"Pearson r = {pearsonr(observed, gse_prediction)[0]:.3f}")
else:
    print("Not enough compounds with melting point, logP and logS to check the GSE.")

A two-parameter equation from 1980, with no fitting whatsoever on our data, already explains a large part of the variance in aqueous solubility. Two things follow. First, the physical coupling between these endpoints is real, so a multitask model has something to exploit. Second, this is the bar for the solubility task: a machine-learned model that does not beat the GSE is not earning its complexity.

#### How many tasks can we afford?

A joint multi-output model needs a dense label matrix, so the compounds it can be trained on are the ones measured for *every* task it covers. Adding a task therefore never adds training data - it can only take it away, and the intersection shrinks multiplicatively.

That makes "which endpoints go in the model" a quantitative decision rather than a matter of taste. Below we count the complete cases for every possible subset of the available endpoints, so that the trade-off between **breadth** (more tasks to share between) and **depth** (more compounds to learn from) is visible.

In [ ]:
from itertools import combinations

subset_rows = []
for size in range(2, len(TASKS) + 1):
    for subset in combinations(TASKS, size):
        n_complete = int(labels[list(subset)].notna().all(axis=1).sum())
        subset_rows.append({
            "n_tasks": size,
            "tasks": ", ".join(t.replace("_", " ") for t in subset),
            "complete cases": n_complete,
            "compound-measurements": n_complete * size,
            "subset": subset,
        })

subset_table = pd.DataFrame(subset_rows).sort_values(
    ["n_tasks", "complete cases"], ascending=[True, False]
)

# the best subset of each size, by number of complete cases
best_per_size = subset_table.groupby("n_tasks").head(1)
best_per_size[["n_tasks", "tasks", "complete cases", "compound-measurements"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 3.6))
ax.plot(best_per_size["n_tasks"], best_per_size["complete cases"],
        marker="o", color="#2a78d6", lw=2, label="compounds available")
ax.plot(best_per_size["n_tasks"], best_per_size["compound-measurements"],
        marker="D", color="#eb6834", lw=2, label="individual measurements used")
ax.set_xlabel("number of tasks in the model")
ax.set_ylabel("count")
ax.set_xticks(best_per_size["n_tasks"])
ax.set_yscale("log")
ax.set_title("Breadth costs depth: the complete-case set shrinks with every task added")
ax.legend(frameon=False)
sns.despine()
plt.show()

Note that the two curves need not point the same way. The number of *compounds* can only fall as tasks are added, but the number of individual *measurements* the model gets to see is that number times the number of tasks, and it can rise — which is exactly the situation in which a joint model is worth building.

Purely numerically, the pair with the largest overlap usually wins this trade-off. We do not follow that here, because the reason these endpoints belong in one model is chemical rather than numerical: melting point, logP and solubility are the three quantities tied together by the General Solubility Equation we checked above. We therefore fix that trio as the default and let the table show what the choice costs, falling back to the best-supported subset if those endpoints are not in your export.

You are encouraged to set `MODEL_TASKS` by hand and re-run the notebook — seeing the results change with the task set is the point of the exercise.

In [ ]:
MIN_COMPLETE_CASES = 400

# The General Solubility Equation trio: the endpoints with a physical reason to
# share a model.
PREFERRED_TASKS = ["melting_point", "logp", "logs"]

available = [t for t in PREFERRED_TASKS if t in TASKS]
if len(available) == len(PREFERRED_TASKS) and \
        labels[available].notna().all(axis=1).sum() >= MIN_COMPLETE_CASES:
    MODEL_TASKS = available
else:
    # fall back to whichever subset gives the most measurements overall
    affordable = subset_table[subset_table["complete cases"] >= MIN_COMPLETE_CASES]
    if len(affordable):
        MODEL_TASKS = list(
            affordable.sort_values("compound-measurements", ascending=False)
            .iloc[0]["subset"]
        )
    else:
        MODEL_TASKS = list(
            subset_table.sort_values("complete cases", ascending=False)
            .iloc[0]["subset"]
        )
    print(f"The preferred trio is not usable here; using {MODEL_TASKS} instead.")

n_complete = int(labels[MODEL_TASKS].notna().all(axis=1).sum())
print(f"modelling these {len(MODEL_TASKS)} tasks: {MODEL_TASKS}")
print(f"complete cases available: {n_complete:,} compounds")

#### Calculate compound descriptors

We reuse the descriptor helper of **Talktorial T032**, with one addition: the ability to *exclude* descriptors by name. RDKit exposes its full 2-D descriptor list through `Descriptors._descList`, a list of `(name, function)` pairs.

Descriptors are the slowest step in the notebook, so we compute them only for the compounds the chosen tasks can actually use: those with at least one measurement among `MODEL_TASKS`. Compounds carrying only, say, a boiling point are irrelevant once boiling point is not being modelled.

In [ ]:
# Descriptors that are themselves models of one of our endpoints. Keeping them in
# the feature matrix would turn the logP task into "learn to correct Crippen".
LEAKY_DESCRIPTORS = {"MolLogP", "MolMR"}


def calculate_molecular_descriptors(smiles, exclude=frozenset()):
    """
    Calculate all 2D RDKit molecular descriptors for a list of SMILES.

    Parameters
    ----------
    smiles : list of str
        Standardised SMILES strings, one per compound.
    exclude : set of str
        Names of RDKit descriptors to leave out of the result.

    Returns
    -------
    pandas.DataFrame
        One row per input SMILES (in the same order), one column per descriptor.
    """
    descriptor_list = [(name, func) for name, func in Descriptors._descList
                       if name not in exclude]

    rows = []
    for smi in tqdm(smiles, desc="descriptors"):
        mol = Chem.MolFromSmiles(smi)
        with warnings.catch_warnings():
            # a few descriptors warn about overflow on large molecules
            warnings.simplefilter("ignore")
            rows.append(
                {name: (func(mol) if mol is not None else np.nan)
                 for name, func in descriptor_list}
            )

    frame = pd.DataFrame(rows, index=smiles)
    return (
        frame.astype(np.float32)
        .replace([np.inf, -np.inf], np.nan)
        .round(4)
    )

In [ ]:
MAX_DESCRIPTOR_COMPOUNDS = 25_000

# Every compound with at least one of the modelled endpoints is a candidate: the
# complete cases train the joint models, and the partially measured ones are used
# later to show what a masked per-task model can do with the rest of the data.
complete_index = labels.index[labels[MODEL_TASKS].notna().all(axis=1)]
partial_index = labels.index[labels[MODEL_TASKS].notna().any(axis=1)].difference(
    complete_index
)

# Complete cases are never dropped; only the partially measured pool is capped.
room_left = max(0, MAX_DESCRIPTOR_COMPOUNDS - len(complete_index))
if len(partial_index) > room_left:
    partial_index = pd.Index(
        rng.choice(partial_index.to_numpy(), size=room_left, replace=False),
        name=partial_index.name,
    )
compound_pool = complete_index.union(partial_index)

print(f"{len(complete_index):,} complete cases + {len(partial_index):,} partially "
      f"measured compounds = {len(compound_pool):,} to describe")

descriptors = calculate_molecular_descriptors(
    structures.loc[compound_pool].tolist(), exclude=LEAKY_DESCRIPTORS
)
descriptors.index = compound_pool          # index by compound, like the labels
print(f"{descriptors.shape[1]} descriptors for {descriptors.shape[0]:,} compounds")
descriptors.iloc[:5, :8]

We also compute Crippen's `MolLogP` separately. It is excluded from the features, but it is the domain baseline our logP model has to beat.

In [ ]:
from rdkit.Chem import Crippen

# `MolFromSmiles` can still return None for an exotic structure, so guard the
# comprehension rather than letting one bad record stop the notebook.
crippen_logp = pd.Series(
    [Crippen.MolLogP(mol) if (mol := Chem.MolFromSmiles(s)) is not None else np.nan
     for s in structures.loc[compound_pool]],
    index=compound_pool,
    name="crippen_logp",
)

if "logp" in MODEL_TASKS:
    measured = labels.loc[compound_pool, "logp"]
    both = measured.notna() & crippen_logp.notna()
    print(f"Crippen MolLogP as a predictor of experimental logP (n = {both.sum():,}):")
    print(f"  R2  = {r2_score(measured[both], crippen_logp[both]):.3f}")
    print(f"  MAE = {mean_absolute_error(measured[both], crippen_logp[both]):.3f} log units")

#### Building the feature matrix

Three pieces of routine hygiene, none of which is optional:

1. **Drop descriptors that failed.** A handful of RDKit descriptors return `NaN` or infinity for unusual structures. We drop any descriptor column that is missing for more than 1 % of compounds, and impute the rest with the column median (computed later, inside the training fold, to avoid leaking test-set statistics - here we just make the matrix finite).
2. **Drop constant descriptors.** A column with zero variance carries no information and breaks some scalers.
3. **Do *not* scale yet.** Scaling must be fitted on the training fold only. We therefore put the scaler inside a `Pipeline` for the models that need one, rather than scaling the whole matrix up front - one of the most common quiet mistakes in QSAR notebooks.

In [ ]:
X_all = descriptors.copy()

missing_fraction = X_all.isna().mean()
X_all = X_all.loc[:, missing_fraction <= 0.01]
X_all = X_all.fillna(X_all.median())
X_all = X_all.loc[:, X_all.std() > 0]

FEATURES = list(X_all.columns)
print(f"feature matrix: {X_all.shape[0]:,} compounds x {len(FEATURES)} descriptors")

Finally we assemble the **modelling set**: the complete cases, i.e. the compounds for which every endpoint is available. As discussed, this is what the joint multi-output models require. We keep the full sparse matrix around as well, because we will come back to it to show what is being given up.

For runtime on a teaching platform we also cap the number of compounds. The binding constraint is the Butina clustering below, which compares every pair of molecules and therefore costs $O(n^2)$ in both time and memory. Increase `MAX_COMPOUNDS` if you have time and RAM to spare; the qualitative conclusions do not depend on it.

In [ ]:
MAX_COMPOUNDS = 4000

model_index = complete_index
if len(model_index) > MAX_COMPOUNDS:
    model_index = pd.Index(
        rng.choice(model_index.to_numpy(), size=MAX_COMPOUNDS, replace=False),
        name=model_index.name,
    )

X = X_all.loc[model_index]
Y = labels.loc[model_index, MODEL_TASKS]
smiles_model = structures.loc[model_index]

print(f"modelling set: {X.shape[0]:,} compounds x {X.shape[1]} descriptors "
      f"-> {Y.shape[1]} outputs")
Y.describe().round(2)

### Data splitting

<img src='images/splitting_methods.png' width="900">

*Figure 4 (repeated):* the four splitting strategies we implement below.

We write one helper per grouping criterion and one generic splitter, so that the modelling code below never has to know which split it is working with.

#### Grouping compounds: scaffolds and similarity clusters

In [ ]:
def bemis_murcko_scaffolds(smiles):
    """
    Assign each molecule to its Bemis-Murcko scaffold.

    The scaffold is the union of the ring systems and the linkers between them,
    with all side chains removed. Acyclic molecules have an empty scaffold, and
    are grouped together under the label ''.

    Parameters
    ----------
    smiles : iterable of str
        SMILES strings.

    Returns
    -------
    numpy.ndarray of str
        One scaffold SMILES per input molecule.
    """
    scaffolds = []
    for smi in smiles:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            scaffolds.append("")
            continue
        try:
            scaffolds.append(MurckoScaffold.MurckoScaffoldSmiles(mol=mol))
        except Exception:
            scaffolds.append("")
    return np.asarray(scaffolds)

In [ ]:
MORGAN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def morgan_fingerprints(smiles):
    """Compute Morgan (ECFP4-like) fingerprints, as in Talktorial T004."""
    fingerprints = []
    for smi in smiles:
        mol = Chem.MolFromSmiles(smi)
        fingerprints.append(MORGAN.GetFingerprint(mol) if mol is not None else None)
    return fingerprints


def butina_clusters(smiles, cutoff=0.4):
    """
    Cluster molecules by fingerprint similarity with the Butina algorithm (T005).

    Parameters
    ----------
    smiles : iterable of str
        SMILES strings.
    cutoff : float
        Distance cutoff; two molecules with a Tanimoto distance below this value
        can end up in the same cluster. Smaller cutoff -> tighter, more clusters.

    Returns
    -------
    numpy.ndarray of int
        One cluster index per input molecule.
    """
    fingerprints = morgan_fingerprints(smiles)
    n = len(fingerprints)

    # Butina wants the lower triangle of the distance matrix as a flat sequence.
    # This is O(n^2): we preallocate a float32 array rather than building a Python
    # list, which for a few thousand compounds is the difference between a few
    # hundred megabytes and a few tens.
    distances = np.empty(n * (n - 1) // 2, dtype=np.float32)
    position = 0
    for i in tqdm(range(1, n), desc="pairwise similarities"):
        similarities = DataStructs.BulkTanimotoSimilarity(
            fingerprints[i], fingerprints[:i]
        )
        distances[position:position + i] = 1.0 - np.asarray(similarities,
                                                            dtype=np.float32)
        position += i

    clusters = Butina.ClusterData(distances, n, cutoff, isDistData=True)

    assignment = np.empty(n, dtype=int)
    for cluster_id, members in enumerate(clusters):
        for member in members:
            assignment[member] = cluster_id
    return assignment

In [ ]:
scaffold_group = bemis_murcko_scaffolds(smiles_model)
cluster_group = butina_clusters(smiles_model, cutoff=0.4)

print(f"{pd.Series(scaffold_group).nunique():,} distinct Bemis-Murcko scaffolds")
print(f"{pd.Series(cluster_group).nunique():,} Butina clusters at a 0.4 distance cutoff")
print(f"largest scaffold group: {pd.Series(scaffold_group).value_counts().iloc[0]:,} compounds")
print(f"largest cluster:        {pd.Series(cluster_group).value_counts().iloc[0]:,} compounds")

Group sizes matter here more than one might expect. In this dataset the single largest Bemis–Murcko "scaffold" is the *empty* one — every acyclic molecule shares it — at roughly a quarter of the compounds, and benzene accounts for another fifth. Either group is on its own larger than a 20 % test set, so a splitter that hands out whole groups at random cannot hit the requested size: one unlucky draw and the test set is twice the size it should be, and the training sets of the different splits are no longer comparable.

The standard remedy, which the splitter below uses, is to assign groups **largest first**: walk the groups from biggest to smallest, putting each into the training set while it still fits, and sending the rest to the test set. The test set then lands close to the requested fraction and — more importantly — it ends up made of the *rare* groups, which is exactly the question a scaffold or cluster split is meant to ask: how does the model do on chemistry it has barely seen? The common chemotypes stay in training, where they belong.

One consequence is worth noting in advance: because the large groups are always assigned the same way, repeating a group split with a different seed varies only the long tail of small groups. The error bars on those splits are therefore narrower than on the random split, and that is a property of the split rather than of the models.

#### The splitter

In [ ]:
def split_train_test(n_samples, split_method, test_size=0.2, seed=SEED,
                     groups=None, stratify_values=None, n_bins=5):
    """
    Split a dataset into a training and a test set.

    Parameters
    ----------
    n_samples : int
        Number of compounds to split.
    split_method : str
        'random'     - every compound assigned independently
        'scaffold'   - whole Bemis-Murcko scaffolds held out (needs `groups`)
        'cluster'    - whole similarity clusters held out (needs `groups`)
        'stratified' - random, but preserving the distribution of
                       `stratify_values` across the two sets
    test_size : float
        Fraction of compounds to place in the test set.
    seed : int
        Random seed, so that a split can be repeated.
    groups : array-like, optional
        Group label per compound, for the group-based methods.
    stratify_values : array-like, optional
        Continuous values to stratify on, for 'stratified'.
    n_bins : int
        Number of quantile bins used to stratify a continuous variable.

    Returns
    -------
    numpy.ndarray of bool
        Test-set mask: True for compounds in the test set.
    """
    generator = np.random.default_rng(seed)
    index = np.arange(n_samples)
    test_mask = np.zeros(n_samples, dtype=bool)

    if split_method == "random":
        chosen = generator.choice(index, size=int(round(test_size * n_samples)),
                                  replace=False)
        test_mask[chosen] = True

    elif split_method in ("scaffold", "cluster"):
        if groups is None:
            raise ValueError(f"'{split_method}' split needs `groups`")
        groups = np.asarray(groups)
        unique_groups, sizes = np.unique(groups, return_counts=True)

        # Largest group first, ties broken at random so that repeats still differ.
        order = np.lexsort((generator.random(len(unique_groups)), -sizes))

        train_capacity = (1.0 - test_size) * n_samples
        n_in_train = 0
        held_out = []
        for position in order:
            size = int(sizes[position])
            if n_in_train + size <= train_capacity:
                n_in_train += size           # this group still fits in training
            else:
                held_out.append(unique_groups[position])
        test_mask = np.isin(groups, held_out)

        if test_mask.all() or not test_mask.any():
            raise ValueError(
                f"the '{split_method}' split could not form both a training and a "
                f"test set: one group covers (almost) the whole dataset. Use a "
                f"different grouping (for the Butina clusters, a smaller `cutoff`)."
            )

    elif split_method == "stratified":
        if stratify_values is None:
            raise ValueError("'stratified' split needs `stratify_values`")
        bins = pd.qcut(pd.Series(np.asarray(stratify_values)), q=n_bins,
                       labels=False, duplicates="drop")
        _, test_index = train_test_split(
            index, test_size=test_size, random_state=seed, stratify=bins
        )
        test_mask[test_index] = True

    else:
        raise ValueError(
            f"Split method '{split_method}' undefined; use "
            f"'random', 'scaffold', 'cluster' or 'stratified'."
        )

    return test_mask

In [ ]:
# The four splits we will use throughout, each as a callable taking a seed.
SPLITS = {
    "random": lambda seed: split_train_test(len(X), "random", seed=seed),
    "scaffold": lambda seed: split_train_test(len(X), "scaffold", seed=seed,
                                              groups=scaffold_group),
    "cluster": lambda seed: split_train_test(len(X), "cluster", seed=seed,
                                             groups=cluster_group),
    "stratified": lambda seed: split_train_test(
        len(X), "stratified", seed=seed, stratify_values=Y[MODEL_TASKS[0]].to_numpy()
    ),
}

for name, make_split in SPLITS.items():
    mask = make_split(SEED)
    print(f"{name:>11}: {(~mask).sum():5,d} train / {mask.sum():5,d} test "
          f"({mask.mean():.1%} test)")

Note that the stratified split above stratifies on **one** endpoint - the first task. With several outputs there is no single thing to stratify on, and that is an honest limitation rather than an oversight: you must choose which endpoint's distribution you care most about preserving, or bin a combination of them.

#### Looking at what each split actually did

In [ ]:
from sklearn.decomposition import PCA

# A 2-D view of the descriptor space, purely for looking at the splits
projection = PCA(n_components=2, random_state=SEED).fit_transform(
    StandardScaler().fit_transform(X)
)

fig, axes = plt.subplots(1, len(SPLITS), figsize=(4.0 * len(SPLITS), 4.0),
                         sharex=True, sharey=True)
for ax, (name, make_split) in zip(axes, SPLITS.items()):
    mask = make_split(SEED)
    ax.scatter(projection[~mask, 0], projection[~mask, 1], s=7, alpha=0.5,
               color="#2a78d6", edgecolor="none", label="training set")
    ax.scatter(projection[mask, 0], projection[mask, 1], s=9, alpha=0.7,
               color="#eb6834", edgecolor="none", marker="D", label="test set")
    ax.set_title(f"{name} split", fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
axes[0].set_ylabel("PC2")
for ax in axes:
    ax.set_xlabel("PC1")
handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="upper center", ncol=2, frameon=False,
           bbox_to_anchor=(0.5, 1.06))
plt.tight_layout()
plt.show()

The random and stratified splits scatter the test compounds evenly through the space; the group-based splits carve out regions. This picture is suggestive but not conclusive - two compounds can be far apart in a PCA of descriptors and still be near-identical molecules. For the real question, we measure similarity directly.

#### Quantifying the leakage

The question a split is really answering is: *for a typical test compound, how similar is the most similar training compound?* If the answer is "almost identical", the test set is measuring memorisation, not generalisation. This is the diagnostic that turns "random splits are optimistic" from a slogan into a number.

In [ ]:
def nearest_neighbour_similarity(fingerprints, test_mask):
    """Maximum Tanimoto similarity from each test compound to any training compound."""
    train_fps = [fp for fp, is_test in zip(fingerprints, test_mask) if not is_test]
    test_fps = [fp for fp, is_test in zip(fingerprints, test_mask) if is_test]
    return np.array([
        max(DataStructs.BulkTanimotoSimilarity(fp, train_fps)) for fp in test_fps
    ])


fingerprints_model = morgan_fingerprints(smiles_model)

similarity_records = []
for name, make_split in SPLITS.items():
    similarities = nearest_neighbour_similarity(fingerprints_model, make_split(SEED))
    similarity_records.append(pd.DataFrame({"split": name, "similarity": similarities}))
similarity_frame = pd.concat(similarity_records, ignore_index=True)

fig, ax = plt.subplots(figsize=(7.5, 4.0))
sns.boxplot(data=similarity_frame, x="split", y="similarity", ax=ax,
            color="#2a78d6", width=0.55, fliersize=2)
ax.set_ylabel("Tanimoto similarity to the\nnearest training compound")
ax.set_xlabel("")
ax.set_title("How much does each split really hold back?")
sns.despine()
plt.show()

similarity_frame.groupby("split")["similarity"].describe()[
    ["mean", "25%", "50%", "75%"]
].round(3)

Read the medians against each other rather than in absolute terms. The random and stratified splits leave every test compound a training neighbour at roughly 0.57 Tanimoto; the scaffold and cluster splits push that down to around 0.44–0.46. The test compounds have become measurably less familiar, and the drop in performance under those splits is the consequence.

Two refinements are worth noticing. The scaffold split achieves the lowest median similarity but the *widest* spread: holding out rare scaffolds produces a test set that is mostly unfamiliar but still contains some compounds with close analogues in training, because two molecules can share no ring system and still be very alike. And the cluster split, which groups on exactly the similarity being measured here, gives the most uniform separation. Neither is "the" right answer — they ask different questions.

**Every performance number below should be read together with this plot**: a score from the random split and a score from the scaffold split are answers to two different questions, and only the second resembles prospective use on new chemistry.

### Multitask regression

#### Helper functions

First the two helpers that everything else is built on: one that fits a model and returns its test-set predictions, and one that turns predictions into a table of per-task metrics.

The fitting helper does the one thing that is easy to forget and fatal to get wrong: **it standardises the outputs**. All our joint models minimise a squared error summed over the output columns, and melting point in °C has a variance thousands of times larger than logP in log units. Without standardisation, "multitask" would quietly mean "melting point, plus four ignored columns". We fit the output scaler on the *training* labels only and invert it on the predictions, so the metrics are reported in the original units.

In [ ]:
def fit_predict(model, X_train, Y_train, X_test):
    """
    Fit a multi-output model on standardised outputs and predict in original units.

    Parameters
    ----------
    model : estimator
        Any estimator accepting a 2D `y`, including the MultiOutput* wrappers.
    X_train, X_test : pandas.DataFrame
        Feature matrices.
    Y_train : pandas.DataFrame
        Training labels, one column per task, no missing values.

    Returns
    -------
    numpy.ndarray of shape (n_test, n_tasks)
        Predictions, on the original scale of each task.
    """
    output_scaler = StandardScaler().fit(Y_train)
    model.fit(X_train, output_scaler.transform(Y_train))
    predictions = np.asarray(model.predict(X_test))
    if predictions.ndim == 1:                      # single-task estimators
        predictions = predictions.reshape(-1, 1)
    return output_scaler.inverse_transform(predictions)

In [ ]:
def per_task_metrics(Y_true, Y_pred, tasks=None):
    """
    Regression metrics for every task separately.

    Note the deliberate absence of an averaged MAE or RMSE: those are in the
    units of their task, and averaging °C with log units is meaningless.

    Parameters
    ----------
    Y_true : pandas.DataFrame or array of shape (n_samples, n_tasks)
    Y_pred : array of shape (n_samples, n_tasks)
    tasks : list of str, optional
        Task names; taken from `Y_true` if it is a DataFrame.

    Returns
    -------
    pandas.DataFrame
        One row per task with Pearson's r, R2, MAE and RMSE.
    """
    if tasks is None:
        tasks = list(Y_true.columns) if hasattr(Y_true, "columns") else \
            [f"task {i}" for i in range(np.shape(Y_true)[1])]
    Y_true = np.asarray(Y_true)
    Y_pred = np.asarray(Y_pred)

    rows = []
    for i, task in enumerate(tasks):
        rows.append({
            "task": task,
            "Pearson r": pearsonr(Y_true[:, i], Y_pred[:, i])[0],
            "R2 score": r2_score(Y_true[:, i], Y_pred[:, i]),
            "MAE": mean_absolute_error(Y_true[:, i], Y_pred[:, i]),
            "RMSE": root_mean_squared_error(Y_true[:, i], Y_pred[:, i]),
        })
    return pd.DataFrame(rows).set_index("task")

In [ ]:
def performance_plot(Y_true, Y_pred, tasks, title=""):
    """
    Observed vs. predicted, one panel per task.

    One panel per task rather than one panel with several colours: the tasks are
    on different scales, so they cannot share an axis, and small multiples keep
    every panel readable.
    """
    Y_true = np.asarray(Y_true)
    Y_pred = np.asarray(Y_pred)

    fig, axes = plt.subplots(1, len(tasks), figsize=(3.2 * len(tasks), 3.4))
    axes = np.atleast_1d(axes)
    for i, (ax, task) in enumerate(zip(axes, tasks)):
        ax.scatter(Y_pred[:, i], Y_true[:, i], s=9, alpha=0.35,
                   color="#2a78d6", edgecolor="none")
        low = min(Y_true[:, i].min(), Y_pred[:, i].min())
        high = max(Y_true[:, i].max(), Y_pred[:, i].max())
        ax.plot([low, high], [low, high], color="#52514e", lw=1.2, ls="--")
        ax.set_xlim(low, high)
        ax.set_ylim(low, high)
        ax.set_title(f"{TASK_LABELS[task].split(' (')[0]}\n"
                     f"$R^2$ = {r2_score(Y_true[:, i], Y_pred[:, i]):.2f}",
                     fontsize=10)
        ax.set_xlabel("predicted")
        if i == 0:
            ax.set_ylabel("observed")
    if title:
        fig.suptitle(title, y=1.04)
    plt.tight_layout()
    plt.show()

#### The models: native, wrapped and boosted

Eight models, organised so that every comparison isolates exactly one thing.

| Model | Strategy | What it is there to show |
|---|---|---|
| `Ridge (native)` | native | a linear reference point |
| `kNN (native)` | native | a non-parametric reference point |
| `Random forest (single-task)` | wrapper around `RandomForestRegressor` | **control** for the native forest |
| `Random forest (native)` | native | the same algorithm, sharing one tree structure |
| `MLP (native)` | native | the textbook shared-hidden-layer multitask network |
| `SVR (MultiOutputRegressor)` | wrapper | a strong single-output model made multi-output |
| `XGBoost (one_output_per_tree)` | boosted, not shared | **control** for the next row |
| `XGBoost (multi_output_tree)` | boosted, shared | one tree per round, vector-valued leaves |

Rows 3 and 4 use *the same* random forest with *the same* hyperparameters; the only difference is whether one tree structure serves all tasks or each task gets its own forest. Rows 7 and 8 are the same comparison inside XGBoost. Any difference within those pairs is the effect of sharing and of nothing else.

Models that are sensitive to feature scale (`Ridge`, `kNN`, `MLP`, `SVR`) are wrapped in a `Pipeline` with a `StandardScaler`, so that the scaler is fitted on the training fold only.

In [ ]:
def build_models(seed=SEED):
    """Build a fresh dictionary of models, so that no state leaks between runs."""
    forest_kwargs = dict(n_estimators=100, min_samples_leaf=2,
                         random_state=seed, n_jobs=-1)
    xgb_kwargs = dict(n_estimators=200, max_depth=6, learning_rate=0.08,
                      subsample=0.9, colsample_bytree=0.8,
                      tree_method="hist", random_state=seed, n_jobs=-1,
                      verbosity=0)

    return {
        # --- estimators with native multi-output support -------------------
        "Ridge (native)": Pipeline([
            ("scale", StandardScaler()),
            ("model", RidgeCV(alphas=np.logspace(-2, 4, 13))),
        ]),
        "kNN (native)": Pipeline([
            ("scale", StandardScaler()),
            ("model", KNeighborsRegressor(n_neighbors=7, weights="distance")),
        ]),
        "Random forest (native)": RandomForestRegressor(**forest_kwargs),
        "MLP (native)": Pipeline([
            ("scale", StandardScaler()),
            ("model", MLPRegressor(hidden_layer_sizes=(128, 64),
                                   early_stopping=True, max_iter=300,
                                   random_state=seed)),
        ]),

        # --- the same algorithms, wrapped: one independent clone per task ---
        "Random forest (single-task)": MultiOutputRegressor(
            RandomForestRegressor(**forest_kwargs)
        ),
        "SVR (MultiOutputRegressor)": Pipeline([
            ("scale", StandardScaler()),
            ("model", MultiOutputRegressor(SVR(C=10.0, epsilon=0.1), n_jobs=-1)),
        ]),

        # --- boosting, with and without a shared tree structure -------------
        "XGBoost (one_output_per_tree)": XGBRegressor(
            multi_strategy="one_output_per_tree", **xgb_kwargs
        ),
        "XGBoost (multi_output_tree)": XGBRegressor(
            multi_strategy="multi_output_tree", **xgb_kwargs
        ),
    }


print("\n".join(build_models().keys()))

> **Note on `multi_strategy`**
>
> `'multi_output_tree'` requires `tree_method='hist'`, which is XGBoost's default in recent versions but is set explicitly above so that the notebook does not depend on that default. The same parameter is available on `XGBClassifier` for multi-label classification, which we use later.

#### A first model, on a single split

Before running everything, let us look at one model on one split in detail - the same way **Talktorial T032** started from a single random-split PCM model.

In [ ]:
test_mask = SPLITS["random"](SEED)
X_train, X_test = X[~test_mask], X[test_mask]
Y_train, Y_test = Y[~test_mask], Y[test_mask]

predictions = fit_predict(
    RandomForestRegressor(n_estimators=100, min_samples_leaf=2,
                          random_state=SEED, n_jobs=-1),
    X_train, Y_train, X_test,
)

metrics = per_task_metrics(Y_test, predictions, MODEL_TASKS)
print("=== Native multi-output random forest, random split ===")
display(metrics.round(3))
performance_plot(Y_test, predictions, MODEL_TASKS)

Compare the `MAE` column with the `median_literature_sd` column of the summary table computed earlier. Where the two are close, the model is predicting about as well as the literature agrees with itself, and the remaining error is not the model's fault.

#### The full comparison: models × splits × repeats

Now the main experiment. Every model is fitted under every split, and every split is repeated with several seeds so that we can tell a real difference from noise.

<div class="alert alert-block alert-warning">

This is the expensive cell of the notebook: `8 models × 4 splits × N_REPEATS` fits. On a two-core Noteable session it runs in a few minutes at the default settings, and roughly linearly longer if you raise `MAX_COMPOUNDS` or add a task. Set `N_REPEATS = 1` for a quick pass - the qualitative picture is already visible then, only the error bars disappear.

</div>

In [ ]:
N_REPEATS = 3
REPEAT_SEEDS = [SEED + i for i in range(N_REPEATS)]

results = []
total = len(SPLITS) * len(REPEAT_SEEDS) * len(build_models())

with tqdm(total=total, desc="fitting") as progress:
    for split_name, make_split in SPLITS.items():
        for seed in REPEAT_SEEDS:
            mask = make_split(seed)
            X_train, X_test = X[~mask], X[mask]
            Y_train, Y_test = Y[~mask], Y[mask]

            for model_name, model in build_models(seed).items():
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    predicted = fit_predict(model, X_train, Y_train, X_test)
                scores = per_task_metrics(Y_test, predicted, MODEL_TASKS).reset_index()
                scores["model"] = model_name
                scores["split"] = split_name
                scores["seed"] = seed
                results.append(scores)
                progress.update(1)

results = pd.concat(results, ignore_index=True)
print(f"{len(results):,} (model, split, seed, task) scores")
results.head()

In [ ]:
# Mean R2 per model and split, averaged over tasks and repeats
overview = (
    results.groupby(["split", "model"])["R2 score"]
    .agg(["mean", "std"])
    .reset_index()
)

split_order = list(SPLITS)
model_order = list(build_models())

fig, axes = plt.subplots(1, len(split_order), figsize=(4.4 * len(split_order), 4.6),
                         sharey=True)
for ax, split_name in zip(axes, split_order):
    subset = overview[overview["split"] == split_name].set_index("model").loc[model_order]
    ax.barh(range(len(subset)), subset["mean"], xerr=subset["std"],
            color="#2a78d6", height=0.62, error_kw={"lw": 1.2, "ecolor": "#52514e"})
    ax.set_yticks(range(len(subset)))
    ax.set_yticklabels(subset.index if ax is axes[0] else [])
    ax.invert_yaxis()
    ax.set_xlabel("mean $R^2$ over tasks")
    ax.set_title(f"{split_name} split")
    ax.axvline(0, color="#52514e", lw=1)
fig.suptitle("Model performance depends on the split at least as much as on the model",
             y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# The same numbers as a table, so that they can be quoted rather than eyeballed.
# Rows are models, columns are splits; each entry is the mean R2 over tasks and
# repeats, with the spread over repeats in brackets.
pivot = overview.assign(
    cell=lambda d: d["mean"].round(3).astype(str) + " (±" + d["std"].round(3).astype(str) + ")"
).pivot(index="model", columns="split", values="cell").loc[model_order, split_order]
display(pivot)

mean_by_split = results.groupby("split")["R2 score"].mean()[split_order]
print("mean R2 over all models and tasks:")
print(mean_by_split.round(3).to_string())

# How big is the choice of split, compared with the choice of model?
print("\ncost of each split, relative to the random split:")
for split_name in split_order:
    if split_name == "random":
        continue
    print(f"  random -> {split_name:<11}: {mean_by_split[split_name] - mean_by_split['random']:+.3f} R2")

print("\nbest minus worst model, within each split:")
for split_name in split_order:
    per_model = results[results["split"] == split_name].groupby("model")["R2 score"].mean()
    print(f"  {split_name:<11}: {per_model.max() - per_model.min():.3f} R2")

Read across the panels before reading within them.

**The split matters, but not uniformly.** Moving from the random split to the cluster split costs only a few hundredths of $R^2$ — the Butina clusters at a 0.4 cutoff are fine-grained, so holding whole clusters out removes near-duplicates without removing whole chemotypes. The **scaffold split is the one that really bites**: because it holds out the *rare* scaffolds, its test set is genuinely unfamiliar chemistry, and the mean $R^2$ falls by a large margin. Compare the printed "cost of each split" against the "best minus worst model" figures underneath: under the scaffold split the choice of split costs several times more than the choice of model, which is the general lesson, while under the cluster split the two are of comparable size.

**Some models do not merely get worse — they become unstable.** Look at the error bars of `Ridge (native)` and `MLP (native)` under the scaffold split. Their spread across repeats is enormous, and the mean $R^2$ can fall below zero, meaning that on some repeats they do worse than predicting the training mean. Extrapolating a linear or neural model into a region of descriptor space it has never seen can go arbitrarily wrong, whereas a tree-based model is bounded by the leaf values it learned and degrades gracefully. That robustness is invisible under a random split and is a large part of why tree ensembles remain the default in QSPR.

**Where the error bars overlap, the ranking is not supported.** The two XGBoost variants sit on top of each other in every panel. Saying "no measurable difference" is the correct reading, and it is a result in its own right.

In [ ]:
# The same results, but per task: where does each model do well?
heatmap_data = (
    results[results["split"] == "cluster"]
    .groupby(["model", "task"])["R2 score"].mean()
    .unstack()[MODEL_TASKS].loc[model_order]
)
heatmap_data.columns = [TASK_LABELS[t].split(" (")[0] for t in heatmap_data.columns]

fig, ax = plt.subplots(figsize=(1.6 * len(MODEL_TASKS) + 4, 0.55 * len(model_order) + 2))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            linewidths=1.5, linecolor="white", cbar_kws={"label": "$R^2$"}, ax=ax)
ax.set_title("Per-task $R^2$ under the cluster split")
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

Averaging over tasks hides exactly what multitask modelling is supposed to be judged on, so this is the view that matters. Read it row by row and compare the pairs that differ by one thing only:

* `Random forest (single-task)` vs `Random forest (native)` - the same forest, with and without a shared tree structure.
* `XGBoost (one_output_per_tree)` vs `XGBoost (multi_output_tree)` - the same booster, with and without shared trees.

A column where the shared variant wins is a task that benefited from the others. A column where it loses is negative transfer, and it is the thing an average would have hidden.

In [ ]:
# The two controlled comparisons, task by task, with their spread over repeats
pairs = [
    ("Random forest (single-task)", "Random forest (native)"),
    ("XGBoost (one_output_per_tree)", "XGBoost (multi_output_tree)"),
]

comparison = results[results["model"].isin([m for pair in pairs for m in pair])]
comparison = comparison[comparison["split"].isin(["cluster", "scaffold"])]

g = sns.catplot(
    data=comparison, kind="bar", x="task", y="R2 score", hue="model",
    col="split", height=4.0, aspect=1.45, errorbar="sd", palette=["#2a78d6", "#eb6834",
                                                                  "#1baf7a", "#eda100"],
    order=MODEL_TASKS,
)
g.set_xticklabels([TASK_LABELS[t].split(" (")[0] for t in MODEL_TASKS], rotation=25,
                  ha="right")
g.set_axis_labels("", "$R^2$")
g.figure.suptitle("Does sharing the model structure help, task by task?", y=1.04)
plt.show()

#### Model selection done properly: cross-validation that matches the split

Everything above compared models on a held-out test set - model *assessment*. Choosing between models or tuning a hyperparameter must instead happen inside the training set, by cross-validation, and the folds should be built the same way the test split was. Here is the difference that makes, on the training set alone.

In [ ]:
mask = SPLITS["cluster"](SEED)
X_train, Y_train = X[~mask], Y[~mask]
groups_train = cluster_group[~mask]

forest = RandomForestRegressor(n_estimators=100, min_samples_leaf=2,
                               random_state=SEED, n_jobs=-1)
output_scaler = StandardScaler().fit(Y_train)
Y_train_scaled = output_scaler.transform(Y_train)

random_cv = cross_val_score(
    forest, X_train, Y_train_scaled,
    cv=KFold(n_splits=5, shuffle=True, random_state=SEED), scoring="r2",
)
grouped_cv = cross_val_score(
    forest, X_train, Y_train_scaled,
    cv=GroupKFold(n_splits=5), groups=groups_train, scoring="r2",
)

print(f"random 5-fold CV on the training set : R2 = {random_cv.mean():.3f} "
      f"(+/- {random_cv.std():.3f})")
print(f"grouped 5-fold CV (cluster groups)   : R2 = {grouped_cv.mean():.3f} "
      f"(+/- {grouped_cv.std():.3f})")

The two numbers estimate the performance of the same model on the same training data, and they disagree - because they are answering different questions. If the final test set holds out whole clusters, then the grouped estimate is the one that is predictive of it, and tuning against the random estimate would tune for the wrong thing. **Match the cross-validation scheme to the split.**

#### Does multitask learning actually help? A learning-curve experiment

The argument for multitask learning is strongest when data are scarce: a task with few measurements borrows structure learned from the others. If that is true here, the gap between the shared and the unshared model should *grow* as the training set shrinks.

We test it directly. Using the cluster split, we fit both variants of the same random forest on 10 %, 25 %, 50 % and 100 % of the training compounds, and score them on the same untouched test set.

In [ ]:
FRACTIONS = [0.1, 0.25, 0.5, 1.0]

curve_rows = []
for seed in REPEAT_SEEDS:
    mask = SPLITS["cluster"](seed)
    X_train_full, Y_train_full = X[~mask], Y[~mask]
    X_test, Y_test = X[mask], Y[mask]
    generator = np.random.default_rng(seed)

    for fraction in FRACTIONS:
        n_keep = min(len(X_train_full), max(50, int(fraction * len(X_train_full))))
        keep = generator.choice(len(X_train_full), size=n_keep, replace=False)
        X_small, Y_small = X_train_full.iloc[keep], Y_train_full.iloc[keep]

        forest_kwargs = dict(n_estimators=100, min_samples_leaf=2,
                             random_state=seed, n_jobs=-1)
        variants = {
            "single-task (wrapped)": MultiOutputRegressor(
                RandomForestRegressor(**forest_kwargs)
            ),
            "multitask (native)": RandomForestRegressor(**forest_kwargs),
        }
        for variant_name, model in variants.items():
            predicted = fit_predict(model, X_small, Y_small, X_test)
            scores = per_task_metrics(Y_test, predicted, MODEL_TASKS).reset_index()
            scores["variant"] = variant_name
            scores["fraction"] = fraction
            scores["n_train"] = n_keep
            scores["seed"] = seed
            curve_rows.append(scores)

curves = pd.concat(curve_rows, ignore_index=True)
curves.head()

In [ ]:
g = sns.relplot(
    data=curves, kind="line", x="n_train", y="R2 score",
    hue="variant", col="task", col_order=MODEL_TASKS,
    marker="o", errorbar="sd", height=3.3, aspect=0.95,
    palette=["#2a78d6", "#eb6834"], facet_kws={"sharey": False},
)
for ax, task in zip(g.axes.flat, MODEL_TASKS):
    ax.set_title(TASK_LABELS[task].split(" (")[0], fontsize=11)
    ax.set_xscale("log")
    ax.set_xlabel("training compounds")
g.set_ylabels("$R^2$ on the held-out clusters")
g.figure.suptitle("Learning curves: does sharing help more when data are scarce?",
                  y=1.05)
plt.show()

In [ ]:
# The same information as a single number per task: how much does sharing add?
delta = (
    curves.pivot_table(index=["task", "n_train"], columns="variant",
                       values="R2 score", aggfunc="mean")
    .assign(**{"gain from sharing": lambda d: d["multitask (native)"]
               - d["single-task (wrapped)"]})
    .round(3)
)
delta.loc[MODEL_TASKS]

Read the `gain from sharing` column down each task, from the smallest training set to the largest. The pattern multitask learning predicts is a positive gain that shrinks as data accumulate — a prior that matters most when there is little evidence.

That is not what this dataset shows. The gains hover around zero and are, if anything, slightly **negative**: sharing one tree structure between melting point, logP and logS costs a little accuracy on every task at almost every training-set size. The effect is small — a couple of hundredths of $R^2$, comparable to the spread across repeats — so the honest summary is "no benefit, and possibly a small penalty", not "a disaster". But it is a genuine negative result, and worth sitting with rather than explaining away.

Why might that be? The General Solubility Equation says these endpoints are *related*, and the earlier check confirmed it. But "related" is not the same as "served by the same splits". The GSE relates the three **values** to each other; it does not say that the molecular features predicting melting point are the features predicting logP. Melting point depends on crystal packing, logP on the distribution of polarity over the surface — a split that separates compounds well for one is not obviously good for the other, and forcing one tree to do both is a compromise rather than a synergy. A model that could share a *representation* while keeping separate output heads — a neural network, as in **Talktorial T022** — is not forced into that compromise in the same way, which is one reason multitask results in the literature are usually reported for networks rather than forests.

#### Using *all* the data: what the wrappers can do that joint models cannot

Everything so far has been fitted on complete cases, because the joint models require a dense label matrix. We measured earlier how expensive that is. Now we can quantify the other side of the trade-off.

The wrapper strategy fits each task independently - so nothing stops us from fitting task $t$ on *every* compound that has a label for task $t$, even if it is missing the other four. Below, a small estimator does exactly that: it is `MultiOutputRegressor` with per-task masking added. It is compared against the joint model on the same test compounds, so the comparison is fair.

One thing to be careful about: the extra training compounds must not include any test compound. We exclude them by `inchikey`, not by row position.

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin, clone


class MaskedMultiOutputRegressor(BaseEstimator, RegressorMixin):
    """
    Like `MultiOutputRegressor`, but tolerating missing labels.

    Clone `estimator` once per output column and fit each clone on the rows that
    actually have a label for that column. This is only possible because the
    clones are independent: a model with a shared structure would need all the
    tasks to agree on which rows exist.

    Parameters
    ----------
    estimator : estimator
        A single-output regressor, cloned once per task.
    """

    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, Y):
        Y = np.asarray(Y, dtype=float)
        self.estimators_ = []
        self.n_fitted_ = []
        for column in range(Y.shape[1]):
            labelled = ~np.isnan(Y[:, column])
            model = clone(self.estimator)
            model.fit(np.asarray(X)[labelled], Y[labelled, column])
            self.estimators_.append(model)
            self.n_fitted_.append(int(labelled.sum()))
        return self

    def predict(self, X):
        X = np.asarray(X)
        return np.column_stack([model.predict(X) for model in self.estimators_])

In [ ]:
# Test set: the same held-out clusters as before, taken from the modelling set
mask = SPLITS["cluster"](SEED)
test_keys = model_index[mask]
X_test, Y_test = X.loc[test_keys], Y.loc[test_keys]

# Training set A: complete cases only (what a joint model can use)
train_keys_dense = model_index[~mask]
X_dense, Y_dense = X_all.loc[train_keys_dense], labels.loc[train_keys_dense, MODEL_TASKS]

# Training set B: every compound with at least one label, minus the test compounds
train_keys_sparse = compound_pool.difference(test_keys)
X_sparse = X_all.loc[train_keys_sparse]
Y_sparse = labels.loc[train_keys_sparse, MODEL_TASKS]

print(f"complete-case training set : {len(X_dense):,} compounds")
print(f"all-data training set      : {len(X_sparse):,} compounds")
print("labels available per task in the all-data training set:")
print(Y_sparse.notna().sum().to_string())

In [ ]:
forest_kwargs = dict(n_estimators=100, min_samples_leaf=2, random_state=SEED, n_jobs=-1)

# (a) joint model, complete cases only
joint_predictions = fit_predict(
    RandomForestRegressor(**forest_kwargs), X_dense, Y_dense, X_test
)
joint_scores = per_task_metrics(Y_test, joint_predictions, MODEL_TASKS)["R2 score"]

# (b) masked wrapper, all available labels. Outputs are standardised per task
#     using only the labels that exist.
output_mean, output_sd = Y_sparse.mean(), Y_sparse.std()
masked = MaskedMultiOutputRegressor(RandomForestRegressor(**forest_kwargs))
masked.fit(X_sparse, (Y_sparse - output_mean) / output_sd)
masked_predictions = masked.predict(X_test) * output_sd.to_numpy() + output_mean.to_numpy()
masked_scores = per_task_metrics(Y_test, masked_predictions, MODEL_TASKS)["R2 score"]

comparison = pd.DataFrame({
    "joint model, complete cases": joint_scores,
    "masked wrapper, all labels": masked_scores,
    "training compounds (joint)": len(X_dense),
    "training compounds (masked)": masked.n_fitted_,
})
comparison.index = [TASK_LABELS[t].split(" (")[0] for t in MODEL_TASKS]
comparison.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 4.0))
positions = np.arange(len(MODEL_TASKS))
width = 0.36
ax.bar(positions - width / 2, comparison.iloc[:, 0], width,
       label="joint model, complete cases", color="#2a78d6")
ax.bar(positions + width / 2, comparison.iloc[:, 1], width,
       label="masked wrapper, all labels", color="#eb6834")
ax.set_xticks(positions)
ax.set_xticklabels(comparison.index, rotation=20, ha="right")
ax.set_ylabel("$R^2$ on the held-out clusters")
ax.set_title("Sharing structure, or keeping the data? The central trade-off")
ax.legend(frameon=False)
sns.despine()
plt.show()

This is the trade-off stated as a measurement rather than as an opinion, and on this dataset it is not close: **keeping the data wins, on every task.** The masked wrapper shares nothing between the endpoints, but it trains melting point on 20,523 compounds instead of 3,200, logP on 12,237 and logS on 6,641 — and it beats the joint model everywhere, by far more than the sharing was ever worth in the learning-curve experiment above.

That ordering is not a universal law; it follows from this dataset's shape. The label matrix is very sparse, so the complete-case intersection is small and the data penalty is severe, while the tasks turn out to transfer poorly, so the compensating benefit is near zero. On a denser matrix of more tightly coupled endpoints the balance could tip the other way. The lesson is the procedure, not the verdict: **measure both sides before committing to a joint model**, because neither "always use multitask" nor "multitask is a gimmick" is a defensible default.

In production the usual resolution is the third option from the theory section — a neural network with a masked loss, which gets the sharing *and* the data, at the cost of leaving scikit-learn's multi-output estimators behind (see **Talktorials T021 and T022** for the neural-network machinery).

### Multitask classification

#### Turning the endpoints into developability flags

Regression is the natural framing for a physicochemical property, but in practice these properties are very often used as *filters*: a compound is flagged as too lipophilic, or too insoluble to formulate, and the question becomes a yes/no one. Framed that way the same dataset becomes a **multi-label classification** problem - each compound carries several binary labels at once - and that is what `MultiOutputClassifier` is for.

The thresholds below are the ones medicinal chemists actually argue about, not arbitrary cut-points:

* **logP > 3** - above this, the risk of promiscuity, poor solubility and metabolic instability rises sharply (the basis of the "3/75" rule and part of Lipinski's rule of five).
* **logS < −4** - roughly 0.1 mM; below this, absorption tends to become solubility-limited.
* **melting point > 200 °C** - "brick dust": crystal packing so stable that dissolution becomes the bottleneck.
* **boiling point > 250 °C** and **vapour pressure < 1 Pa** (i.e. `log(Pa) < 0`) — low volatility, which matters for handling, analysis and environmental fate. Note that OCHEM delivers vapour pressure in log(Pa), so the threshold is expressed in those units rather than the log(mmHg) of the older literature.

Binarising a measured continuous value does throw information away, and it is worth being explicit that a compound just either side of a threshold is being treated as categorically different when it is not. We do it here because the decision really is binary; if you only have the continuous values, modelling them and thresholding the *prediction* is usually the better route.

In [ ]:
THRESHOLDS = {
    "logp": (3.0, "greater", "lipophilic (logP > 3)"),
    "logs": (-4.0, "less", "poorly soluble (logS < -4)"),
    "melting_point": (200.0, "greater", "high-melting (MP > 200 °C)"),
    "boiling_point": (250.0, "greater", "low-volatility (BP > 250 °C)"),
    "vapour_pressure": (0.0, "less", "low-volatility (vapour pressure < 1 Pa)"),
}

flag_frames = {}
for task in MODEL_TASKS:
    threshold, direction, label = THRESHOLDS[task]
    values = Y[task]
    flag_frames[label] = (values > threshold) if direction == "greater" else (values < threshold)

Y_binary = pd.DataFrame(flag_frames, index=Y.index).astype(int)

prevalence = Y_binary.mean().rename("fraction positive").to_frame()
prevalence["n positive"] = Y_binary.sum()

# A threshold that almost nothing crosses gives a label no model can learn, and
# every metric then reports the same uninformative number. Drop those labels,
# loudly, rather than filling the plots with zeros.
MIN_POSITIVES = 25
too_rare = prevalence.index[prevalence["n positive"] < MIN_POSITIVES]
for flag in too_rare:
    print(f"dropping '{flag}': only {int(prevalence.loc[flag, 'n positive'])} "
          f"positives, fewer than MIN_POSITIVES = {MIN_POSITIVES}")
Y_binary = Y_binary.drop(columns=too_rare)
FLAGS = list(Y_binary.columns)

if len(FLAGS) < 2:
    print("Fewer than two usable flags survived the domain thresholds; falling back "
          "to a median split of every endpoint so that the multi-label machinery "
          "below still has something to work with.")
    Y_binary = pd.DataFrame(
        {f"above-median {TASK_LABELS[task].split(' (')[0].lower()}":
            (Y[task] > Y[task].median()).astype(int) for task in MODEL_TASKS},
        index=Y.index,
    )
    FLAGS = list(Y_binary.columns)

prevalence.round(3)

Check the prevalences before going further. A label that is positive for 2 % of compounds is a different - and much harder - problem from one that is balanced, and it is the reason we report Matthews correlation coefficient and balanced accuracy rather than plain accuracy: a model that answers "negative" to everything would score 98 % accuracy on such a label while being completely useless.

Below a certain point the label stops being learnable at all, and every model returns the same uninformative score. The cell above therefore drops any flag with fewer than `MIN_POSITIVES` positives rather than carrying it into the plots. If a threshold you care about is dropped, that is itself the finding: this dataset does not contain enough examples on the far side of it, and the answer is more data rather than a better model.

#### Three multi-label classifiers: `MultiOutputClassifier`, native multi-label forests, XGBoost

In [ ]:
def build_classifiers(seed=SEED):
    """Multi-label classifiers, one per strategy from Figure 2."""
    return {
        # native multi-label: one forest, leaves hold a distribution per label
        "Random forest (native multi-label)": RandomForestClassifier(
            n_estimators=200, min_samples_leaf=2, class_weight="balanced_subsample",
            random_state=seed, n_jobs=-1,
        ),
        # wrapper: one independent clone per label
        "HistGB (MultiOutputClassifier)": MultiOutputClassifier(
            HistGradientBoostingClassifier(max_iter=200, random_state=seed), n_jobs=-1,
        ),
        "SVC (MultiOutputClassifier)": Pipeline([
            ("scale", StandardScaler()),
            ("model", MultiOutputClassifier(
                SVC(C=5.0, probability=True, class_weight="balanced",
                    random_state=seed), n_jobs=-1)),
        ]),
        # boosted, with and without a shared tree structure
        "XGBoost (one_output_per_tree)": XGBClassifier(
            multi_strategy="one_output_per_tree", n_estimators=200, max_depth=6,
            learning_rate=0.08, tree_method="hist", random_state=seed,
            n_jobs=-1, verbosity=0,
        ),
        "XGBoost (multi_output_tree)": XGBClassifier(
            multi_strategy="multi_output_tree", n_estimators=200, max_depth=6,
            learning_rate=0.08, tree_method="hist", random_state=seed,
            n_jobs=-1, verbosity=0,
        ),
    }


print("\n".join(build_classifiers().keys()))

> **A gotcha worth knowing**
>
> The three families disagree about what `predict_proba` returns for a multi-label problem.
> `MultiOutputClassifier` and a natively multi-label `RandomForestClassifier` both return a **list of `T` arrays**, each of shape `(n_samples, 2)` - you have to take column 1 of each. `XGBClassifier` with a 2-D `y` returns a **single array of shape `(n_samples, T)`** already containing the positive-class probabilities. The helper below normalises both into one `(n_samples, T)` array, which is the kind of small compatibility shim that multi-output code always ends up needing.

In [ ]:
def positive_class_probabilities(model, X):
    """
    Return positive-class probabilities as an (n_samples, n_labels) array.

    Handles both conventions: a list of per-label (n_samples, 2) arrays, and a
    single (n_samples, n_labels) array.
    """
    probabilities = model.predict_proba(X)
    if isinstance(probabilities, list):
        return np.column_stack([p[:, 1] for p in probabilities])
    probabilities = np.asarray(probabilities)
    if probabilities.ndim == 3:            # (n_labels, n_samples, 2)
        return probabilities[:, :, 1].T
    return probabilities

In [ ]:
def multilabel_metrics(Y_true, Y_pred, Y_score, flags):
    """
    Per-label and aggregate metrics for a multi-label classifier.

    Returns
    -------
    per_label : pandas.DataFrame
        MCC, balanced accuracy and ROC-AUC for each label.
    aggregate : dict
        Hamming loss and subset accuracy over all labels at once.
    """
    Y_true = np.asarray(Y_true)
    Y_pred = np.asarray(Y_pred)

    rows = []
    for i, flag in enumerate(flags):
        row = {
            "label": flag,
            "MCC": matthews_corrcoef(Y_true[:, i], Y_pred[:, i]),
            "balanced accuracy": balanced_accuracy_score(Y_true[:, i], Y_pred[:, i]),
        }
        # ROC-AUC is undefined if only one class is present in the test set
        row["ROC-AUC"] = (roc_auc_score(Y_true[:, i], Y_score[:, i])
                          if len(np.unique(Y_true[:, i])) == 2 else np.nan)
        rows.append(row)

    aggregate = {
        "Hamming loss": hamming_loss(Y_true, Y_pred),
        "subset accuracy": accuracy_score(Y_true, Y_pred),
    }
    return pd.DataFrame(rows).set_index("label"), aggregate

In [ ]:
mask = SPLITS["cluster"](SEED)
X_train, X_test = X[~mask], X[mask]
Yb_train, Yb_test = Y_binary[~mask], Y_binary[mask]

classification_results = []
aggregate_results = []

for name, model in tqdm(build_classifiers().items(), desc="classifiers"):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(X_train, Yb_train)
        predicted = np.asarray(model.predict(X_test))
        scores = positive_class_probabilities(model, X_test)

    per_label, aggregate = multilabel_metrics(Yb_test, predicted, scores, FLAGS)
    per_label = per_label.reset_index()
    per_label["model"] = name
    classification_results.append(per_label)
    aggregate_results.append({"model": name, **aggregate})

classification_results = pd.concat(classification_results, ignore_index=True)
pd.DataFrame(aggregate_results).set_index("model").round(3)

The two aggregate numbers say different things. **Hamming loss** is the fraction of individual label decisions that were wrong - a forgiving, per-label view. **Subset accuracy** requires *every* label of a compound to be right, and it collapses quickly as the number of labels grows: with five labels each at 90 % accuracy and independent errors, only about 59 % of compounds would be entirely correct. When a screening cascade applies several filters in sequence, subset accuracy is the number that describes it.

In [ ]:
g = sns.catplot(
    data=classification_results, kind="bar", x="label", y="MCC", hue="model",
    height=4.4, aspect=2.1, palette=["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                                     "#e87ba4"],
)
g.set_xticklabels(rotation=22, ha="right")
g.set_axis_labels("", "Matthews correlation coefficient")
g.figure.suptitle("Multi-label classification under the cluster split", y=1.04)
g.ax.axhline(0, color="#52514e", lw=1)
plt.show()

classification_results.pivot(index="label", columns="model", values="MCC").round(3)

MCC is the right headline here: it is a correlation coefficient between predicted and true labels, it returns 0 for a model that is no better than chance regardless of the class balance, and it cannot be inflated by predicting the majority class. Balanced accuracy and ROC-AUC are in the table for comparison; ROC-AUC in particular looks reassuringly high even for labels where MCC shows the model has learned very little, because it only measures *ranking* and never has to commit to a threshold.

In [ ]:
classification_results.set_index(["model", "label"]).round(3)

## Discussion

We set out to do what **Talktorial T032** did for several protein targets, but for several molecular properties, and with the tasks moved from the input side of the model to the output side. The headline result is a negative one, which is worth stating plainly before anything else: **on this dataset, sharing a model between the endpoints did not help.**

**On the three implementations.** All three produce an `(n_samples, n_tasks)` prediction array, and from the outside they are interchangeable. Inside they are not:

* The `MultiOutputRegressor` / `MultiOutputClassifier` wrappers make any single-output estimator multi-output, but they share nothing. Their value is convenience and, for us, the fact that they are the *control* against which sharing can be measured.
* Estimators with native support build one structure for all tasks. That is where transfer can happen — and also where negative transfer can happen.
* XGBoost's `multi_strategy` exposes exactly this choice as a parameter, which makes it the cleanest way to run the experiment: change one string, change nothing else. The two variants finished indistinguishably here, within the spread across repeats, while `multi_output_tree` produced a model a third of the size.

**On whether multitask helped.** No, and the experiments say so in two independent ways. The controlled comparisons (the same random forest, the same booster, with and without a shared structure) showed differences inside the repeat-to-repeat noise. The learning curves, which should be where a benefit appears if it appears anywhere, showed gains hovering at or slightly below zero at every training-set size. And when we stopped insisting on complete cases, the model that shares *nothing* but sees six times as much data beat the joint model on every endpoint. For this task set, the data is worth more than the sharing — by a wide margin.

This does not refute multitask learning; it locates it. The endpoints are physically coupled — the General Solubility Equation reaches $R^2 = 0.65$ on our compounds with no fitting at all — yet coupled *values* did not translate into shared *features*. The conditions under which multitask pays off are narrower than the enthusiasm around it suggests, and they are checkable in advance: a dense label matrix, tasks that are scarce individually, and a model class that can share a representation without forcing every split to be a compromise.

**On the splits.** The scaffold split cost several times more $R^2$ than the gap between the best and worst model, and it also exposed something a random split hides entirely: `Ridge` and `MLP` do not just degrade on unfamiliar chemistry, they become unstable, with mean $R^2$ falling below zero on some repeats. A model comparison run only on a random split would have reported those two as competitive. That is the argument for several splits in one sentence.

**On the label matrix.** The tension between sharing and data is the practical heart of multitask modelling on public data. Joint models want a dense `Y`; public data give a very sparse one — here 35 % filled, and only 1,489 of 45,272 compounds carry all five endpoints — and the complete-case intersection shrinks as tasks are added. Choosing a task set is therefore a quantitative decision about breadth against depth, which is why the notebook makes it explicitly rather than by habit.

**Things we did not do, which a real study would.**

* *No hyperparameter tuning.* Every model used sensible defaults. A fair comparison tunes each model inside the training folds, with a cross-validation scheme matching the split.
* *Descriptors only.* Fingerprints, or a learned representation, might change the ranking — particularly for melting point, where the limitation is arguably that our descriptors say nothing about crystal packing at all.
* *No neural multitask model.* The one architecture that can share a representation *and* use the sparse labels, via a masked loss, is exactly the one scikit-learn's multi-output estimators cannot express. Building it on top of **Talktorial T022** is the natural follow-up, and the obvious way to test the explanation offered above.
* *No applicability domain.* We reported average performance over a whole test set, not per-prediction confidence. A deployed model should say when it is being asked about chemistry it has not seen — and the scaffold-split results show what happens when it cannot.
* *Aggregation hides disagreement.* We reduced several literature records per compound to a median. The `value_std` column says how much those records disagreed, and modelling that uncertainty explicitly — for example by weighting well-measured compounds more heavily — is a natural extension.

## Quiz

1. In **Talktorial T032**, the protein target entered the model as a block of input features. In this talktorial the properties are outputs instead. Name one thing the PCM formulation can do that the multitask formulation cannot, and one thing the multitask formulation can do that the PCM formulation cannot.
2. `MultiOutputRegressor(RandomForestRegressor())` and `RandomForestRegressor()` fitted on a 2-D `y` both return predictions of the same shape. What is the difference between them, and which one can transfer information between tasks?
3. What exactly does `multi_strategy='multi_output_tree'` change about how XGBoost grows a tree, and why does that make the model smaller?
4. Why must the outputs be standardised before fitting a joint multi-output model on melting point (°C) and logP (log units)? What would a model trained without standardisation effectively be predicting?
5. Neither scikit-learn nor XGBoost accepts `NaN` in `y`. Give two strategies for training on a sparse label matrix anyway, and state what each one costs.
6. Why is the nearest-neighbour similarity plot a more convincing diagnostic of a split than the PCA scatter plot?
7. A model reaches $R^2 = 0.82$ under a random split and $R^2 = 0.52$ under a scaffold split. Which number should go into the abstract of a paper proposing it as a prospective tool, and why?
8. `MolLogP` was deliberately removed from the feature matrix. Explain why, and say whether leaving it in would have been *leakage* in the strict sense.
9. For a label that is positive for only 3 % of compounds, a classifier reaches 97 % accuracy. What is its likely MCC, and what does that tell you?
10. Under what circumstances would you expect multitask learning to make one of the tasks *worse* than a dedicated single-task model?
11. Under the scaffold split, `Ridge` and `MLP` had very large error bars and a mean $R^2$ near or below zero, while the tree ensembles degraded smoothly. What does that difference say about how the two kinds of model behave outside their training domain?
12. The General Solubility Equation shows that melting point, logP and logS are physically coupled, yet sharing a tree structure between them did not improve any of the three. How can both of those statements be true at once?